In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers datasets scikit-learn seaborn matplotlib wordcloud nltk



In [ ]:
import torch
print(torch.cuda.is_available())


In [ ]:
import pandas as pd
import re
import torch

#libraries for data explore
import seaborn as sns
import matplotlib.pyplot as plt

#libraries for Model training and evaluation
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [102]:
from google.colab import files
uploaded = files.upload()


KeyboardInterrupt: 

In [ ]:
# Example: Load a CSV dataset
df = pd.read_csv("turkish_phishing_dataset.csv")


df.shape

In [ ]:
# Randomly sample 7504,000 rows from the full dataset
df = pd.read_csv("turkish_phishing_dataset.csv")
#df = df.sample(n=7000, random_state=42).reset_index(drop=True)

print(df.head())

# Preprocessing Steps Not Applied (with Justifications)

- **Remove URLs** : URLs are strong indicators of phishing, so we keep them as features.
- **Remove special characters** : Symbols like @, !, $, etc., are commonly used in phishing tactics and should be retained.
- **Remove punctuation** : Punctuation may be part of deceptive formatting or URLs, so we do not remove it.
- **Remove numbers** : Numbers (like fake invoice IDs or OTPs) may signal phishing intent, so we preserve them.
- **Remove stopwords** : Common words may carry phishing signals in certain contexts, so we keep them for now.
- **Remove capital letters** : Capitalization (e.g., “URGENT”) often indicates phishing, so we retain letter casing.
- **Stemming or Lemmatization** : We keep words in their original form to preserve possible phishing-related phrasing.
- **Remove HTML tags** : Some phishing emails use HTML tricks; we want to analyze them if present.
- **Spelling correction** : Typos and misspellings may be deliberate in phishing emails, so we avoid correcting them.



In [ ]:
print(df.columns)


In [ ]:
# Remove nulls
df.dropna(inplace=True)

# Kategori kolonunu string olarak bırak
print(df['Kategori'].unique())
# Örn: ['Güvenilir', 'Oltalama']

# Map string -> int
mapping = {"Güvenilir": 0, "Oltalama": 1}
df['Kategori'] = df['Kategori'].map(mapping)

# Lowercasing İçerik
def clean_text(text):
    return str(text).lower()
df['İçerik'] = df['İçerik'].apply(lambda x: str(x).lower())
df['İçerik'] = df['İçerik'].apply(clean_text)


In [ ]:
print(df.info())

In [ ]:
df.shape

In [ ]:
df.tail()

# Explore the dataset

In [ ]:
print(df['Kategori'].value_counts())


#### Plot Class Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Kategori')  # doğru kolon adı
plt.title('Oltalama vs Güvenilir Emailler')
plt.xticks([0, 1], ['Güvenilir', 'Oltalama'])
plt.ylabel('Count')
plt.show()


#### Email/Text special character Length Distribution by Class
This graph is a boxplot comparing the number of special characters used in legitimate vs phishing emails.

Special characters: ! @ # $ % ^ & * ( ) _ + = { } [ ] : ; " ' < > / ? , . ~ etc.


In [ ]:
# Special character count
df['special_chars'] = df['İçerik'].apply(
    lambda x: sum(not c.isalnum() and not c.isspace() for c in str(x))
)

plt.figure(figsize=(8,5))
sns.boxplot(x='Kategori', y='special_chars', data=df, hue='Kategori', palette='coolwarm', dodge=False)

plt.xticks([0, 1], ['Güvenilir', 'Oltalama'])
plt.title('Special Characters Count by Class')
plt.xlabel('Kategori')
plt.ylabel('Special Characters Count')
plt.grid(True)
plt.show()


#### Most Common Words

In [ ]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Türkçe stopword listesi
stop_words = set(stopwords.words('turkish'))

# Sadece "Oltalama" olanları birleştir
phishing_text = ' '.join(df[df['Kategori'] == 1]['İçerik']).lower().split()

# Stopword ve alfasayısal filtre
filtered_words = [word for word in phishing_text if word.isalpha() and word not in stop_words]

# En sık geçen 30 kelime
word_freq = Counter(filtered_words).most_common(30)

# Barplot
words, counts = zip(*word_freq)
plt.figure(figsize=(10,6))
sns.barplot(x=list(counts), y=list(words), hue=list(counts), palette='magma', legend=False)
plt.title('Oltalama Maillerde En Sık Geçen 20 Türkçe Kelime')
plt.xlabel('Frekans')
plt.ylabel('Kelime')
plt.show()


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Oltalama maillerden kelimeleri birleştir
wordcloud_text = ' '.join(filtered_words)

# Türkçe karakter desteği için font_path ekleyebilirsin (örneğin DejaVuSans)
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='magma'
).generate(wordcloud_text)


# Görselleştir
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Oltalama Maillerde Kelime Bulutu')
plt.show()


 ### Split the Dataset in training , validation and test

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Önce %20 test seti ayır
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    df['İçerik'].tolist(),       # metinler
    df['Kategori'].tolist(),     # etiketler (0 = Güvenilir, 1 = Oltalama)
    test_size=0.2,
    random_state=42,
    stratify=df['Kategori']      # sınıf dengesini koru
)

# 2. Kalan %80'i %70 train, %10 validation olarak ayır
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=0.125,  # toplamın %10'u
    random_state=42,
    stratify=train_val_labels
)


#### Tokenization using BERT Tokenizer

These encodings are the tokenized versions of your input texts, structured in a way that BERT can understand

In [ ]:
tokenizer = BertTokenizer.from_pretrained("dbmdz/bert-base-turkish-uncased")
# Tokenize the datasets
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)


In [ ]:
print(train_encodings['input_ids'][0])          # View input IDs of the first text
print(train_encodings['attention_mask'][0])

#### Convert to HuggingFace Dataset

A custom PyTorch dataset class for handling tokenized email text data and labels.

- **Inherits**: `torch.utils.data.Dataset`
- **Purpose**: Feeds input to Hugging Face `Trainer` in a structured format.

#### Methods:
- `__init__(self, encodings, labels)`
  - Stores tokenized input data (`encodings`) and labels.
- `__getitem__(self, idx)`
  - Returns the input tensors and label for a specific index.
  - Example output:
    ```python
    {
      'input_ids': tensor(...),
      'attention_mask': tensor(...),
      'labels': tensor(...)
    }
    ```
- `__len__(self)`
  - Returns the total number of samples.

    

#### Data input format for bert
- input_ids: Token IDs representing the input text.

- attention_mask: Binary mask to distinguish real tokens (1) from padding (0).

- labels: True class values used for supervised learning.

- [CLS]: Special token added at the start of input for classification tasks.

- [SEP]: Separator token marking the end of a sentence or segment.



In [ ]:
class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)


#### Load Pretrained BERT for Classification

In [ ]:
model = BertForSequenceClassification.from_pretrained("dbmdz/bert-base-turkish-uncased", num_labels=2)

#### Define Evaluation Metrics

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


#### Training Arguments

`TrainingArguments` is a configuration class that defines how your model should be trained. You pass it key training parameters such as:

- `output_dir`: Where to save model checkpoints.
- `learning_rate`: Controls how fast the model learns.
- `num_train_epochs`: Number of complete passes through the training data.
- `evaluation_strategy`: When to run validation (e.g., `"epoch"` means after every epoch).
- `per_device_train_batch_size`: Batch size per GPU/CPU device during training.
- `logging_dir`: Directory to store training logs.

It does **not** train the model — it only stores training settings that the `Trainer` will use.



In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    report_to="none",           # wandb loglamayı kapatır
    eval_strategy="epoch",      # senin sürümde doğru parametre bu
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    optim="adamw_torch",
    logging_dir='./logs',
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)


#### Trainer




`Trainer` is the training engine provided by Hugging Face. It handles the full model training lifecycle. Specifically, it:

- Loads and trains your model.
- Handles batching, shuffling, and tokenization.
- Trains the model using PyTorch under the hood.
- Evaluates the model on your `eval_dataset`.
- Uses the settings defined in `TrainingArguments`.
- Automatically supports training on GPU or TPU if available.

`Trainer` simplifies the process so you don’t need to manually write training loops, backpropagation, or optimizer logic.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

#### Train the BERT Model

The `trainer.train()` method is the command that **starts the training process** using the configuration and datasets you’ve defined with Hugging Face’s `Trainer` class.

It does the following:

- Loads the `train_dataset` and batches the data.
- Feeds each batch through the model.
- Computes the loss and performs backpropagation to update model weights.
- Evaluates the model on the `eval_dataset` if `evaluation_strategy` is set (e.g., `"epoch"`).
- Saves checkpoints if `save_strategy` is configured.
- Logs training metrics (loss, accuracy, etc.) if `logging_dir` is specified.

##### What It Uses Internally:
- Model defined in `Trainer(model=...)`
- Training configuration from `TrainingArguments`
- Tokenizer and data collator for handling padding and batching
- Evaluation function if `compute_metrics` is provided

##### Output:
After running `trainer.train()`, the model will be:
- Trained on your training data
- Optionally validated each epoch
- Saved to the directory specified in `output_dir`


In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)


## Plot Accuracy and Loss vs Epoch

In [ ]:
import matplotlib.pyplot as plt

# Extract logs
log_history = trainer.state.log_history

# Gather data
train_loss = [entry['loss'] for entry in log_history if 'loss' in entry]
eval_loss = [entry['eval_loss'] for entry in log_history if 'eval_loss' in entry]
eval_acc  = [entry['eval_accuracy'] for entry in log_history if 'eval_accuracy' in entry]
epochs = list(range(1, len(eval_loss)+1))

# Plot loss
plt.figure(figsize=(10,4))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, label='Training Loss')
plt.plot(epochs, eval_loss, label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, eval_acc, marker='o', color='green', label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy over Epochs")
plt.legend()

plt.tight_layout()
plt.show()


## Save the Final Model & Tokenizer

In [ ]:
# Save model and tokenizer
model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")


In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer
model = BertForSequenceClassification.from_pretrained("./final_model")
tokenizer = BertTokenizer.from_pretrained("./final_model")


### Test the model

##### Tokenize test data

In [ ]:
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

#### Convert to PyTorch Dataset

In [ ]:
test_dataset = EmailDataset(test_encodings, test_labels)

#### Evaluate Model on Test Set

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Run predictions on the test set
test_results = trainer.predict(test_dataset)

# Extract predicted class labels
y_pred = test_results.predictions.argmax(axis=1)
y_true = test_labels  # Make sure test_labels is a list of integers

# Print all core metrics
print("Final Test Evaluation Metrics:\n")
print("Accuracy       :", accuracy_score(y_true, y_pred))
print("Precision      :", precision_score(y_true, y_pred))
print("Recall         :", recall_score(y_true, y_pred))
print("F1 Score       :", f1_score(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["Legitimate", "Phishing"]))



In [ ]:
# @title
# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report

y_pred = test_results.predictions.argmax(axis=1)
print(classification_report(test_labels, y_pred, target_names=["Legitimate", "Phishing"]))


In [ ]:
model.save_pretrained("final_model")
tokenizer.save_pretrained("final_model")


In [ ]:
!zip -r final_model.zip final_model


In [ ]:
from google.colab import files
files.download("final_model.zip")
